# MobileNetV3 Attack Sanity Check
Diagnostic notebook to verify whether 100% PGD ASR is genuine or a bug.
Checks: (1) clean accuracy, (2) confidence, (3) budget usage, (4) logit margins, (5) per-class ASR.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, Subset
import pandas as pd
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

DATA_DIR    = 'dataset'
TEST_CSV    = os.path.join(DATA_DIR, 'Test.csv')
NUM_CLASSES = 43
BATCH_SIZE  = 64
EVAL_SUBSET = 2000
SEED        = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CKPT = {
    'MobileNetV3': 'best_mobilenetv3_gtsrb.pth',
    'ResNet-50':   'best_resnet50_gtsrb.pth',
    'VGG-16':      'best_vgg16_gtsrb.pth',
}

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device: {device}')


## Data & Model Setup

In [ ]:
class GTSRBTestDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['Path'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['ClassId'])
        if self.transform:
            image = self.transform(image)
        return image, label


test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_test_dataset = GTSRBTestDataset(TEST_CSV, DATA_DIR, transform=test_transform)
print(f'Full test set: {len(full_test_dataset)} images')

torch.manual_seed(SEED)
indices = torch.randperm(len(full_test_dataset))[:EVAL_SUBSET].tolist()
eval_dataset = Subset(full_test_dataset, indices)

full_loader = DataLoader(full_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset,      batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


def build_mobilenetv3():
    m = models.mobilenet_v3_large(weights=None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
    return m

def build_resnet50():
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def build_vgg16():
    m = models.vgg16(weights=None)
    m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    return m

BUILDERS = {
    'MobileNetV3': build_mobilenetv3,
    'ResNet-50':   build_resnet50,
    'VGG-16':      build_vgg16,
}

nets = {}
for name, builder in BUILDERS.items():
    m = builder()
    state = torch.load(CKPT[name], map_location=device)
    m.load_state_dict(state)
    m = m.to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    nets[name] = m
    print(f'Loaded {name}')

criterion = nn.CrossEntropyLoss()
colors = {'MobileNetV3': '#e74c3c', 'ResNet-50': '#2ecc71', 'VGG-16': '#3498db'}


In [ ]:
def pgd_attack(model, images, labels, eps, alpha, steps):
    x = images.clone() + torch.empty_like(images).uniform_(-eps, eps)
    x = torch.clamp(x, images - eps, images + eps).detach()
    for _ in range(steps):
        x = x.requires_grad_(True)
        loss = criterion(model(x), labels)
        loss.backward()
        x = (x + alpha * x.grad.sign()).detach()
        x = torch.clamp(x, images - eps, images + eps)
    return x

print('PGD defined.')


## Check 1 — Clean Accuracy Verification

In [ ]:
def eval_accuracy(model, loader):
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total, total

mnv3 = nets['MobileNetV3']

full_acc, full_n = eval_accuracy(mnv3, full_loader)
print(f'MobileNetV3 FULL test set ({full_n} images): {full_acc*100:.2f}%')
print(f'  (Expected ~95.10% from training notebook)')
print()

print(f'{"Model":<14} {"Subset Acc":>12}  (N={EVAL_SUBSET}, seed={SEED})')
print('-' * 30)
subset_accs = {}
for name, m in nets.items():
    acc, n = eval_accuracy(m, eval_loader)
    subset_accs[name] = acc
    print(f'{name:<14} {acc*100:>11.2f}%')

print()
diff = abs(full_acc - subset_accs['MobileNetV3']) * 100
status = 'PASS' if diff < 2.0 else 'FAIL'
print(f'CHECK 1 {status} -- Full vs subset delta = {diff:.2f}pp (checkpoint loading {"OK" if diff < 2.0 else "suspicious"})')


## Check 2 — Prediction Confidence Analysis

In [ ]:
conf_data = {}

for name, m in nets.items():
    top_confs = []
    with torch.no_grad():
        for imgs, labels in eval_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            probs = F.softmax(m(imgs), dim=1)
            preds = probs.argmax(1)
            mask  = (preds == labels)
            top_confs.append(probs.max(1).values[mask].cpu())
    conf_data[name] = torch.cat(top_confs).numpy()

fig, ax = plt.subplots(figsize=(10, 5))
for name, confs in conf_data.items():
    ax.hist(confs, bins=60, alpha=0.55, color=colors[name], label=name, density=True)
ax.set_xlabel('Top-1 Softmax Confidence (correctly classified images)')
ax.set_ylabel('Density')
ax.set_title('Confidence Distribution -- 2000-image Eval Subset')
ax.legend()
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150)
plt.show()
print('Saved: confidence_distribution.png')

print()
print(f'{"Model":<14} {"Mean":>8} {"Median":>8} {"Min":>8} {"Max":>8}')
print('-' * 50)
for name, confs in conf_data.items():
    print(f'{name:<14} {confs.mean():>8.4f} {np.median(confs):>8.4f} {confs.min():>8.4f} {confs.max():>8.4f}')

print()
mnv3_conf_mean  = conf_data['MobileNetV3'].mean()
other_conf_mean = np.mean([conf_data[k].mean() for k in ['ResNet-50', 'VGG-16']])
if mnv3_conf_mean < other_conf_mean - 0.03:
    print(f'CHECK 2 NOTE -- MobileNetV3 mean confidence ({mnv3_conf_mean:.4f}) is lower '
          f'than other models ({other_conf_mean:.4f}), consistent with weaker margins.')
else:
    print(f'CHECK 2 NOTE -- Confidence levels are comparable across models.')


## Check 3 — Perturbation Budget Usage (epsilon=0.05)

In [ ]:
EPS_B   = 0.05
ALPHA_B = EPS_B / 4
STEPS_B = 20

budget_data = {}

for name, m in nets.items():
    norms = []
    fracs = []
    print(f'Running PGD budget analysis for {name}...', end=' ', flush=True)
    for imgs, labels in eval_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            clean_mask = (m(imgs).argmax(1) == labels)
        if clean_mask.sum() == 0:
            continue
        adv = pgd_attack(m, imgs, labels, EPS_B, ALPHA_B, STEPS_B)
        with torch.no_grad():
            fooled_mask = clean_mask & (m(adv).argmax(1) != labels)
        if fooled_mask.sum() == 0:
            continue
        pert = (adv - imgs).abs().view(imgs.size(0), -1).max(1).values
        for i in range(len(labels)):
            if fooled_mask[i]:
                norms.append(pert[i].item())
                fracs.append(pert[i].item() / EPS_B)
    budget_data[name] = {'norms': np.array(norms), 'fracs': np.array(fracs)}
    print(f'{len(norms)} fooled images')

print()
print(f'{"Model":<14} {"Mean Linf":>10} {"Median Linf":>12} {"Min":>8} {"Max":>8} {"Mean %budget":>14}')
print('-' * 70)
for name, d in budget_data.items():
    n, f = d['norms'], d['fracs']
    if len(n) == 0:
        print(f'{name:<14}  no fooled images')
        continue
    print(f'{name:<14} {n.mean():>10.4f} {np.median(n):>12.4f} {n.min():>8.4f} {n.max():>8.4f} {f.mean()*100:>13.1f}%')

print()
mnv3_fracs = budget_data.get('MobileNetV3', {}).get('fracs', np.array([]))
if len(mnv3_fracs) > 0:
    mnv3_bud = mnv3_fracs.mean()
    ref_buds = [budget_data[k]['fracs'].mean() for k in ['ResNet-50', 'VGG-16']
                if len(budget_data.get(k, {}).get('fracs', [])) > 0]
    if ref_buds:
        ref_mean = np.mean(ref_buds)
        if mnv3_bud < ref_mean - 0.05:
            print(f'CHECK 3 NOTE -- MobileNetV3 uses only {mnv3_bud*100:.1f}% of budget '
                  f'vs {ref_mean*100:.1f}% for other models. Decision boundaries are closer.')
        else:
            print(f'CHECK 3 NOTE -- Budget usage is comparable across models '
                  f'({mnv3_bud*100:.1f}% vs {ref_mean*100:.1f}%).')


## Check 4 — Logit Margin Analysis

In [ ]:
margin_data = {}

for name, m in nets.items():
    margins = []
    with torch.no_grad():
        for imgs, labels in eval_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = m(imgs)
            mask   = (logits.argmax(1) == labels)
            if mask.sum() == 0:
                continue
            top2 = logits[mask].topk(2, dim=1).values
            margins.append((top2[:, 0] - top2[:, 1]).cpu())
    margin_data[name] = torch.cat(margins).numpy()

fig, ax = plt.subplots(figsize=(10, 5))
for name, marg in margin_data.items():
    ax.hist(marg, bins=80, alpha=0.55, color=colors[name], label=name, density=True)
ax.axvline(1.0, color='k', linestyle='--', linewidth=1, label='margin = 1.0')
ax.set_xlabel('Logit Margin (top logit - 2nd logit)')
ax.set_ylabel('Density')
ax.set_title('Logit Margin Distribution -- 2000-image Eval Subset')
ax.legend()
plt.tight_layout()
plt.savefig('margin_distribution.png', dpi=150)
plt.show()
print('Saved: margin_distribution.png')

print()
print(f'{"Model":<14} {"Mean":>8} {"Median":>8} {"Min":>8} {"Max":>8} {"% < 1.0":>9}')
print('-' * 60)
for name, marg in margin_data.items():
    pct_low = (marg < 1.0).mean() * 100
    print(f'{name:<14} {marg.mean():>8.3f} {np.median(marg):>8.3f} {marg.min():>8.3f} {marg.max():>8.3f} {pct_low:>8.1f}%')

print()
mnv3_med_marg = np.median(margin_data['MobileNetV3'])
ref_med_margs = [np.median(margin_data[k]) for k in ['ResNet-50', 'VGG-16']]
if mnv3_med_marg < np.mean(ref_med_margs) - 1.0:
    print(f'CHECK 4 NOTE -- MobileNetV3 median margin ({mnv3_med_marg:.3f}) is substantially '
          f'lower than other models ({np.mean(ref_med_margs):.3f}). '
          f'Predictions are much closer to the decision boundary.')
else:
    print(f'CHECK 4 NOTE -- Margins are in a similar range across models.')


## Check 5 — Per-Class ASR Breakdown (MobileNetV3, epsilon=0.01)

In [ ]:
EPS_CLS   = 0.01
ALPHA_CLS = EPS_CLS / 4
STEPS_CLS = 20

class_correct = np.zeros(NUM_CLASSES, dtype=int)
class_fooled  = np.zeros(NUM_CLASSES, dtype=int)

print(f'Running PGD at eps={EPS_CLS} on MobileNetV3...', flush=True)
for imgs, labels in eval_loader:
    imgs, labels = imgs.to(device), labels.to(device)
    with torch.no_grad():
        clean_mask = (mnv3(imgs).argmax(1) == labels)
    adv = pgd_attack(mnv3, imgs, labels, EPS_CLS, ALPHA_CLS, STEPS_CLS)
    with torch.no_grad():
        fooled_mask = clean_mask & (mnv3(adv).argmax(1) != labels)
    for i in range(len(labels)):
        c = labels[i].item()
        if clean_mask[i]:
            class_correct[c] += 1
            if fooled_mask[i]:
                class_fooled[c] += 1

class_asr = np.where(class_correct > 0, class_fooled / class_correct, float('nan'))

print(f'\nPer-class ASR at eps={EPS_CLS} (MobileNetV3):')
print(f'{"Class":>6} {"N correct":>10} {"N fooled":>9} {"ASR":>8}')
print('-' * 38)
for c in range(NUM_CLASSES):
    if class_correct[c] > 0:
        marker = ' <-- <100%' if class_asr[c] < 1.0 else ''
        print(f'{c:>6} {class_correct[c]:>10} {class_fooled[c]:>9} {class_asr[c]*100:>7.1f}%{marker}')

valid    = ~np.isnan(class_asr)
pct_full = (class_asr[valid] >= 1.0).mean() * 100
mean_asr = np.nanmean(class_asr) * 100
print(f'\nClasses at 100% ASR: {(class_asr[valid] >= 1.0).sum()} / {valid.sum()} ({pct_full:.1f}%)')
print(f'Mean per-class ASR:  {mean_asr:.2f}%')


## Final Verdict

In [ ]:
print('=' * 72)
print('  SANITY CHECK SUMMARY')
print('=' * 72)

# Check 1
full_acc_pct = full_acc * 100
sub_acc_pct  = subset_accs['MobileNetV3'] * 100
c1 = abs(full_acc_pct - sub_acc_pct) < 2.0
print(f'\n[Check 1] Clean accuracy')
print(f'  Full test ({full_n} images): {full_acc_pct:.2f}%')
print(f'  Eval subset (2000):         {sub_acc_pct:.2f}%')
print(f'  Status: {"PASS" if c1 else "FAIL"}')

# Check 2
mnv3_conf_val  = conf_data['MobileNetV3'].mean()
other_conf_val = np.mean([conf_data[k].mean() for k in ['ResNet-50', 'VGG-16']])
c2_lower = mnv3_conf_val < other_conf_val - 0.03
print(f'\n[Check 2] Mean top-1 confidence')
for name, confs in conf_data.items():
    print(f'  {name:<14}: {confs.mean():.4f}')
print(f'  MobileNetV3 confidence is {"LOWER" if c2_lower else "COMPARABLE"}')

# Check 3
mnv3_fracs_v = budget_data.get('MobileNetV3', {}).get('fracs', np.array([]))
c3_lower = False
if len(mnv3_fracs_v) > 0:
    mnv3_bud_v = mnv3_fracs_v.mean()
    ref_buds_v = [budget_data[k]['fracs'].mean() for k in ['ResNet-50', 'VGG-16']
                  if len(budget_data.get(k, {}).get('fracs', [])) > 0]
    c3_lower   = mnv3_bud_v < np.mean(ref_buds_v) - 0.05 if ref_buds_v else False
    print(f'\n[Check 3] Mean PGD budget used (eps={EPS_B})')
    for name, d in budget_data.items():
        if len(d['fracs']) > 0:
            print(f'  {name:<14}: {d["fracs"].mean()*100:.1f}% of epsilon budget')
    print(f'  MobileNetV3 uses {"LESS" if c3_lower else "SIMILAR"} budget')

# Check 4
mnv3_med_v   = np.median(margin_data['MobileNetV3'])
ref_med_v    = [np.median(margin_data[k]) for k in ['ResNet-50', 'VGG-16']]
c4_lower     = mnv3_med_v < np.mean(ref_med_v) - 1.0
print(f'\n[Check 4] Median logit margin')
for name, marg in margin_data.items():
    pct_low = (marg < 1.0).mean() * 100
    print(f'  {name:<14}: median={np.median(marg):.3f}, % margin<1.0 = {pct_low:.1f}%')
print(f'  MobileNetV3 margins are {"SMALLER" if c4_lower else "COMPARABLE"}')

# Check 5
c5_universal = pct_full > 90
print(f'\n[Check 5] Per-class ASR at eps={EPS_CLS} (MobileNetV3)')
print(f'  Classes at 100% ASR: {(class_asr[valid] >= 1.0).sum()} / {valid.sum()} ({pct_full:.1f}%)')
print(f'  Mean per-class ASR:  {mean_asr:.2f}%')

evidence = sum([c1, c2_lower, c4_lower, c5_universal])

print()
print('=' * 72)
if c1 and evidence >= 2:
    print('  VERDICT: 100% ASR is GENUINE')
    print()
    reasons = []
    if c1:           reasons.append('checkpoint loads correctly (full/subset accuracy match)')
    if c2_lower:     reasons.append('MobileNetV3 softmax confidence is lower (weaker predictions)')
    if c4_lower:     reasons.append('MobileNetV3 logit margins are smaller (closer to decision boundary)')
    if c5_universal: reasons.append(f'{pct_full:.0f}% of classes are at 100% ASR (universal fragility)')
    for r in reasons:
        print(f'  + {r}')
else:
    print('  VERDICT: 100% ASR is SUSPICIOUS -- investigate further')
    if not c1:
        print('  ! Check 1 failed: accuracy mismatch suggests checkpoint loading issue')
print('=' * 72)
